# In-Class Activity: How a CNN Reads Remote-Sensing Data

**Dataset:** Landsat land-cover image patches  
**Model:** A small convolutional neural network (CNN) in TensorFlow/Keras  

## Learning goals

By the end of this activity, you should be able to:

1. Explain how a 3 × 3 × 4 remote-sensing patch differs from an ordinary row of tabular data.
2. Apply a convolutional filter to a spectral band and interpret the resulting feature map.
3. Describe local connectivity, shared weights, activation functions, epochs, dropout, and early stopping.
4. Train a small multiclass CNN without leaking validation or test information.
5. Compare the CNN with multinomial logistic regression.
6. Interpret learning curves, a confusion matrix, learned filters, and internal feature maps.

## 1. Why this dataset can be treated as imagery

Each row contains **36 predictor values**:

$$9\text{ pixels} \times 4\text{ spectral bands}=36\text{ features}.$$

The nine pixels form a 3 × 3 neighborhood. A conventional model receives a flat vector of 36 numbers. A CNN instead receives an array with shape:

$$\text{height} \times \text{width} \times \text{channels}=3\times3\times4.$$

The CNN therefore knows which values are spatial neighbors. The target is the land-cover class of the center pixel.

The original class numbers are 1, 2, 3, 4, 5, and 7; class 6 does not occur in this dataset.

In [ ]:
# If needed, install packages in the active notebook kernel:
# %pip install tensorflow pandas numpy matplotlib seaborn scikit-learn

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 7

# These must be set before TensorFlow is imported.
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def reset_random_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

reset_random_seeds()

sns.set_theme(style="whitegrid")
print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

## 2. Load and inspect the Landsat data

The loader first looks for a local copy and otherwise reads the course GitHub file. Keeping a local copy is recommended for class.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/Landsat_Landcover.csv"
landsat = pd.read_csv(DATA_URL)
print("Shape:", landsat.shape)
display(landsat.head())

In [ ]:
CLASS_NAMES = {
    1: "red soil",
    2: "cotton crop",
    3: "grey soil",
    4: "damp grey soil",
    5: "soil with vegetation stubble",
    7: "very damp grey soil",
}

landsat["land_cover"] = landsat["land_cover"].astype(int)
class_counts = (
    landsat["land_cover"]
    .value_counts()
    .sort_index()
    .rename_axis("class_number")
    .reset_index(name="n")
)
class_counts["land_cover_name"] = class_counts["class_number"].map(CLASS_NAMES)
display(class_counts)

ax = sns.barplot(data=class_counts, x="land_cover_name", y="n", color="steelblue")
ax.set(title="Land-cover class distribution", xlabel="", ylabel="Observations")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 3. Reshape a row into a four-band image patch

The columns are ordered by pixel first and spectral band second. We reshape each row to `(3, 3, 4)`. TensorFlow uses **channels last**, so the final dimensions are `(height, width, channels)`.

In [ ]:
feature_columns = [
    f"pixel_{pixel}_band_{band}"
    for pixel in range(1, 10)
    for band in range(1, 5)
]

X_flat = landsat[feature_columns].to_numpy(dtype=np.float32)
y_original = landsat["land_cover"].to_numpy()

class_numbers = np.array(sorted(CLASS_NAMES))
class_to_index = {class_number: i for i, class_number in enumerate(class_numbers)}
index_to_class = {i: class_number for class_number, i in class_to_index.items()}
y = np.array([class_to_index[value] for value in y_original], dtype=np.int64)

# (samples, 36) -> (samples, height=3, width=3, bands=4)
X_image = X_flat.reshape(-1, 3, 3, 4)

print("Flat input shape:", X_flat.shape)
print("CNN input shape:", X_image.shape)
print("Target shape:", y.shape)

In [ ]:
# Visualize the four spectral bands for one observation.
sample_index = 12
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

vmin = X_image[sample_index].min()
vmax = X_image[sample_index].max()

for band_index, ax in enumerate(axes):
    sns.heatmap(
        X_image[sample_index, :, :, band_index],
        annot=True,
        fmt=".2f",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        square=True,
        ax=ax,
    )
    ax.set_title(f"Band {band_index + 1}")
    ax.set_xticks([])
    ax.set_yticks([])

class_number = index_to_class[y[sample_index]]
fig.suptitle(f"One 3 × 3 Landsat patch: {CLASS_NAMES[class_number]}", y=1.05)
plt.tight_layout()
plt.show()

## 4. See a convolution happen

A kernel is a small matrix of weights. It slides across the image, multiplies local values by its weights, and adds the products. The same weights are reused at every location—this is **weight sharing**.

For a 3 × 3 input and a 2 × 2 kernel with no padding, the output is a 2 × 2 feature map:

$$\text{output size}=3-2+1=2.$$

In [ ]:
# A simple hand-designed vertical-contrast kernel.
# tf.nn.conv2d expects inputs as (batch, height, width, channels)
# and filters as (kernel_height, kernel_width, input_channels, output_channels).
input_band = tf.convert_to_tensor(
    X_image[sample_index, :, :, 0][None, :, :, None],
    dtype=tf.float32,
)
kernel_2d = np.array(
    [[1.0, -1.0],
     [1.0, -1.0]],
    dtype=np.float32,
)
kernel = tf.convert_to_tensor(kernel_2d[:, :, None, None])

feature_map = tf.nn.conv2d(
    input_band,
    filters=kernel,
    strides=1,
    padding="VALID",
).numpy()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
sns.heatmap(input_band.numpy()[0, :, :, 0], annot=True, fmt=".2f",
            cmap="coolwarm", center=0, cbar=False, square=True, ax=axes[0])
axes[0].set_title("Input: band 1")

sns.heatmap(kernel_2d, annot=True, fmt=".0f", cmap="coolwarm", center=0,
            cbar=False, square=True, ax=axes[1])
axes[1].set_title("2 × 2 kernel")

sns.heatmap(feature_map[0, :, :, 0], annot=True, fmt=".2f",
            cmap="coolwarm", center=0, cbar=False, square=True, ax=axes[2])
axes[2].set_title("Output feature map")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

### Calculate one output by hand

Use the upper-left 2 × 2 portion of the input and the kernel above:

$$z=(x_{11})(1)+(x_{12})(-1)+(x_{21})(1)+(x_{22})(-1).$$

**TODO**  
1. Confirm that your value matches the upper-left cell of the output feature map.
2. What local pattern produces a large positive value?

In [ ]:
z = #TODO: Complete the hand calculation here
print(z)

### Change the kernal

Change the kernel to `[[1, 1], [-1, -1]]`.  

**TODO**  
What direction of contrast does it detect?

## 5. Split before learning anything from the data

We use three partitions:

- **Training set:** fit weights and calculate normalization statistics.
- **Validation set:** monitor training and choose when to stop.
- **Test set:** estimate final performance once, after modeling decisions are complete.

The test set must not be used to choose epochs, hidden units, dropout, or any other setting.

In [ ]:
all_indices = np.arange(len(y))

train_idx, remaining_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

val_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=y[remaining_idx],
)

print(f"Training observations:   {len(train_idx)}")
print(f"Validation observations: {len(val_idx)}")
print(f"Test observations:       {len(test_idx)}")

In [ ]:
# Compute one mean and standard deviation per band using TRAINING DATA ONLY.
band_mean = X_image[train_idx].mean(axis=(0, 1, 2), keepdims=True)
band_std = X_image[train_idx].std(axis=(0, 1, 2), keepdims=True)
band_std = np.where(band_std == 0, 1.0, band_std)

X_scaled = (X_image - band_mean) / band_std

print("Training-band means after scaling:", X_scaled[train_idx].mean(axis=(0, 1, 2)).round(4))
print("Training-band SDs after scaling:  ", X_scaled[train_idx].std(axis=(0, 1, 2)).round(4))

### Leakage check

In [ ]:
# Create explicit arrays for Keras. Keep the test set separate until final evaluation.
X_train = X_scaled[train_idx]
y_train = y[train_idx]

X_val = X_scaled[val_idx]
y_val = y[val_idx]

X_test = X_scaled[test_idx]
y_test = y[test_idx]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

## 6. Baseline: multinomial logistic regression

Logistic regression receives the same 36 values, but flattened into a vector. It can assign a separate coefficient to every position and band, but it does not explicitly learn a reusable local pattern.

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)

logistic_model.fit(X_flat[train_idx], y[train_idx])

logistic_val_pred = logistic_model.predict(X_flat[val_idx])
logistic_test_pred = logistic_model.predict(X_flat[test_idx])
logistic_test_prob = logistic_model.predict_proba(X_flat[test_idx])

logistic_results = {
    "model": "Logistic regression",
    "validation_accuracy": accuracy_score(y[val_idx], logistic_val_pred),
    "test_accuracy": accuracy_score(y[test_idx], logistic_test_pred),
    "test_balanced_accuracy": balanced_accuracy_score(y[test_idx], logistic_test_pred),
    "test_log_loss": log_loss(y[test_idx], logistic_test_prob),
}

pd.DataFrame([logistic_results])

## 7. Build a small CNN

The network has these stages:

1. **Convolution:** 12 learned 2 × 2 filters examine all four bands.
2. **ReLU:** negative responses become zero, adding nonlinearity.
3. **Flatten:** the feature maps become a vector.
4. **Dense hidden layer:** combines detected local patterns.
5. **Dropout:** randomly masks some hidden activations during training.
6. **Output layer:** uses softmax to return six class probabilities.

We do not use pooling because a 3 × 3 patch is already extremely small.

In [ ]:
def build_cnn(n_classes=6, n_filters=12, dropout=0.25, learning_rate=0.01):
    inputs = keras.Input(shape=(3, 3, 4), name="landsat_patch")

    x = layers.Conv2D(
        filters=n_filters,
        kernel_size=(2, 2),
        activation="relu",
        name="convolution",
    )(inputs)
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(24, activation="relu", name="hidden")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    outputs = layers.Dense(
        n_classes,
        activation="softmax",
        name="class_probabilities",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="landsat_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

keras.backend.clear_session()
reset_random_seeds(SEED)

cnn = build_cnn(
    n_classes=len(class_numbers),
    n_filters=12,
    dropout=0.25,
    learning_rate=0.01,
)

cnn.summary()
print("Trainable parameters:", cnn.count_params())

### Architecture questions

**TODO**  
1. Why does the convolution produce 12 feature maps?
2. Why is each feature map 2 × 2?
3. What does `12 * 2 * 2` represent in the first dense layer?
4. During evaluation, dropout is turned off. Why?

## 8. Train over epochs with early stopping

An **epoch** is one complete pass through the training set. The optimizer updates the weights after every mini-batch.

Early stopping stores the weights from the epoch with the lowest validation loss. Training stops when validation loss fails to improve for several epochs. This limits wasted computation and helps prevent overfitting.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=7,
    min_delta=1e-4,
    restore_best_weights=True,
    verbose=1,
)

history_object = cnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=0,
)

history = pd.DataFrame(history_object.history)
history.insert(0, "epoch", np.arange(1, len(history) + 1))
display(history.tail())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

history.plot(x="epoch", y=["loss", "val_loss"], ax=axes[0])
axes[0].set(title="Loss by epoch", ylabel="Cross-entropy loss")

history.plot(x="epoch", y=["accuracy", "val_accuracy"], ax=axes[1])
axes[1].set(title="Accuracy by epoch", ylabel="Accuracy", ylim=(0, 1))

plt.tight_layout()
plt.show()

### Read the learning curves

**TODO**  
1. Around which epoch was validation loss lowest?
3. What pattern would indicate overfitting?
4. Why should we select the stopping point using validation loss rather than test accuracy?

## 9. Evaluate the untouched test set

We now use the test set for the first time. Keras automatically disables dropout when `predict()` or `evaluate()` runs.

In [ ]:
# Predict class probabilities, then convert them to class indices.
cnn_test_prob = cnn.predict(X_test, verbose=0)
cnn_test_pred = cnn_test_prob.argmax(axis=1)

best_epoch_index = history["val_loss"].idxmin()

cnn_results = {
    "model": "Small CNN",
    "validation_accuracy": history.loc[best_epoch_index, "val_accuracy"],
    "test_accuracy": accuracy_score(y_test, cnn_test_pred),
    "test_balanced_accuracy": balanced_accuracy_score(y_test, cnn_test_pred),
    "test_log_loss": log_loss(y_test, cnn_test_prob),
}

model_comparison = pd.DataFrame([logistic_results, cnn_results])
display(model_comparison.round(3))

In [ ]:
display_names = [CLASS_NAMES[number] for number in class_numbers]
cm = confusion_matrix(y_test, cnn_test_pred, labels=np.arange(len(class_numbers)))

plt.figure(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_names,
    yticklabels=display_names,
)
plt.title("CNN confusion matrix: final test set")
plt.xlabel("Predicted class")
plt.ylabel("Observed class")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    cnn_test_pred,
    target_names=display_names,
    digits=3,
))

### Model comparison

1. Which model has higher balanced accuracy?
2. Which classes are most frequently confused?
3. Does the CNN clearly outperform logistic regression? Give possible reasons as to why this occurs. 

> A CNN is not automatically better. Its advantage depends on whether local spatial structure contains useful signal, whether the patch is large enough, and whether the amount of training data supports the added flexibility.

## 10. Visualize what the CNN learned

The first convolutional layer contains 12 filters. Each filter has a separate 2 × 2 weight matrix for each of the four input bands. The model combines all four band-specific responses to create one output feature map.

In [ ]:
# Keras stores Conv2D kernels as:
# (kernel_height, kernel_width, input_channels, output_filters)
learned_filters = cnn.get_layer("convolution").get_weights()[0]

# Show the band-1 weights for all 12 learned filters.
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
max_abs = np.abs(learned_filters[:, :, 0, :]).max()

for filter_index, ax in enumerate(axes.ravel()):
    sns.heatmap(
        learned_filters[:, :, 0, filter_index],
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-max_abs,
        vmax=max_abs,
        cbar=False,
        square=True,
        ax=ax,
    )
    ax.set_title(f"Filter {filter_index + 1}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Learned 2 × 2 filter weights for spectral band 1", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Create a model whose output is the first convolutional layer.
activation_model = keras.Model(
    inputs=cnn.input,
    outputs=cnn.get_layer("convolution").output,
)

example_position = 0
activated_maps = activation_model.predict(
    X_test[example_position:example_position + 1],
    verbose=0,
)[0]  # shape: (2, 2, 12)

fig, axes = plt.subplots(3, 4, figsize=(8, 6))
vmax = activated_maps.max()

for map_index, ax in enumerate(axes.ravel()):
    sns.heatmap(
        activated_maps[:, :, map_index],
        annot=True,
        fmt=".2f",
        cmap="magma",
        vmin=0,
        vmax=vmax,
        cbar=False,
        square=True,
        ax=ax,
    )
    ax.set_title(f"Map {map_index + 1}")
    ax.set_xticks([])
    ax.set_yticks([])

true_class_number = index_to_class[y_test[example_position]]
fig.suptitle(
    f"ReLU feature maps for one test patch: {CLASS_NAMES[true_class_number]}",
    y=1.02,
)
plt.tight_layout()
plt.show()

### Interpret the internals carefully

- A bright feature-map cell means that a learned local pattern activated strongly at that location.
- A dark cell may represent either a weak response or a negative response removed by ReLU.
- Individual filters do not necessarily have simple human-readable meanings.
- Learned filters and feature maps help explain computation, but they are not complete causal explanations.

## 11. One-setting-at-a-time experiments

Do **not** use the test set for these experiments. Change one setting, refit from a fresh model, and compare the validation curves.

### Experiment A: learning rate

Refit with `learning_rate=0.001` and then `learning_rate=0.05`.

### Experiment B: dropout

Refit with `dropout=0.0` and then `dropout=0.50`.

### Experiment C: number of filters

Call `build_cnn()` with `n_filters=4` or `n_filters=24`. Keras infers the dense-layer input size automatically.

In [1]:
# Starter cell for a controlled experiment.
# Change only one setting at a time, then rerun.

keras.backend.clear_session()
reset_random_seeds(SEED)

EXPERIMENT_DROPOUT = 0.50
EXPERIMENT_LEARNING_RATE = 0.01

experiment_model = build_cnn(
     n_classes=len(class_numbers),
     n_filters=12,
     dropout=EXPERIMENT_DROPOUT,
     learning_rate=EXPERIMENT_LEARNING_RATE)
experiment_history = experiment_model.fit(
     X_train,
     y_train,
     validation_data=(X_val, y_val),
     epochs=50,
     batch_size=128,
     callbacks=[keras.callbacks.EarlyStopping(
         monitor="val_loss",
         patience=7,
         restore_best_weights=True,
     )],
     verbose=0,
 )

NameError: name 'keras' is not defined

## 12. Final Questions

Answer in a few sentences each.

1. What information does the CNN use that logistic regression does not explicitly encode?
2. What is shared across image locations in a convolutional layer, and why is that useful?
3. Explain the difference between a learned filter and a feature map.
4. Based on the evidence from this activity, is a CNN justified for these data? Defend your answer using at least two metrics or plots.